# Rebuild Meta + Google Gold and Unified (natural keys)

Rebuilds Meta gold from `Files/Silver/meta_ads`, refreshes Google gold, materializes `Gold.rpt_unified_ad_performance` with **both** platforms.

Uses business keys only (`account_id`, `campaign_id`, ...) — **no** `*_sk` columns.



In [ ]:
WORKSPACE_ID = "718e8176-5d40-4a9c-88ff-50ac97ac49ba"
LAKEHOUSE_ID = "981fbe98-2f01-41d8-bf2f-a85e5cd9e2a2"
BASE = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}"
META_SILVER = f"{BASE}/Files/Silver/meta_ads"
GOOGLE_SILVER = f"{BASE}/Files/Development/Silver/GoogleAds"
GOLD_FILES_META = f"{BASE}/Files/Gold"
GOLD_FILES_GOOGLE = f"{BASE}/Files/Development/Gold/GoogleAds"
GOLD_FILES_UNIFIED = f"{BASE}/Files/Development/Gold"
S = "Gold"
print("[RUNTIME]", META_SILVER)



In [ ]:
from pyspark.sql import functions as F

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {S}")
spark.conf.set("spark.sql.parquet.vorder.default", "true")

def write_managed_and_files(df, table, files_path, partition_cols=None):
    target = f"{S}.{table}"
    w = df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    if partition_cols:
        w = w.partitionBy(*partition_cols)
    w.saveAsTable(target)
    fw = df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    if partition_cols:
        fw = fw.partitionBy(*partition_cols)
    fw.save(files_path)
    n = spark.table(target).count()
    print(f"[OK] {target} + files -> {n:,} rows")
    return n

campaigns = spark.read.format("delta").load(f"{META_SILVER}/silver_meta_campaigns")
adsets = spark.read.format("delta").load(f"{META_SILVER}/silver_meta_adsets")
ads = spark.read.format("delta").load(f"{META_SILVER}/silver_meta_ads")
insights = spark.read.format("delta").load(f"{META_SILVER}/silver_meta_ad_insights")
print("[META silver]", campaigns.count(), adsets.count(), ads.count(), insights.count())

dim_account = (
    campaigns.select("account_id", "account_name", F.lit("meta").alias("platform"), "tenant_id")
    .dropDuplicates(["account_id"])
    .withColumn("gold_processed_at", F.current_timestamp())
)
dim_campaign = (
    campaigns.select(
        "campaign_id", "account_id", "campaign_name",
        F.col("objective"), F.col("status"),
        F.col("daily_budget").alias("daily_budget_inr"),
        F.col("budget_remaining").alias("budget_remaining_inr"),
    ).dropDuplicates(["campaign_id"]).withColumn("gold_processed_at", F.current_timestamp())
)
dim_adset = (
    adsets.select(
        "adset_id", "campaign_id", "account_id", "adset_name", "status", "optimization_goal",
        F.col("daily_budget").alias("daily_budget_inr"),
        F.col("lifetime_budget").alias("lifetime_budget_inr"),
    ).dropDuplicates(["adset_id"]).withColumn("gold_processed_at", F.current_timestamp())
)
_ad_cols = ["ad_id", "adset_id", "campaign_id", "account_id", "ad_name", "status"]
if "effective_status" in ads.columns:
    dim_ad = ads.select(*_ad_cols, "effective_status")
else:
    dim_ad = ads.select(*_ad_cols).withColumn("effective_status", F.lit(None).cast("string"))
dim_ad = dim_ad.dropDuplicates(["ad_id"]).withColumn("gold_processed_at", F.current_timestamp())

rpt_meta = (
    insights.alias("i")
    .withColumn("full_date", F.to_date(F.col("i.date_start")))
    .join(dim_account.alias("a"), F.col("i.account_id") == F.col("a.account_id"), "left")
    .join(dim_campaign.alias("c"), F.col("i.campaign_id") == F.col("c.campaign_id"), "left")
    .join(dim_adset.alias("s"), F.col("i.adset_id") == F.col("s.adset_id"), "left")
    .join(dim_ad.alias("ad"), F.col("i.ad_id") == F.col("ad.ad_id"), "left")
    .where(F.col("full_date").isNotNull())
    .select(
        F.col("full_date"),
        F.year("full_date").alias("year"),
        F.month("full_date").alias("month"),
        F.date_format("full_date", "MMMM").alias("month_name"),
        F.date_format("full_date", "EEEE").alias("day_name"),
        F.col("i.account_id").alias("account_id"),
        F.coalesce(F.col("a.account_name"), F.col("i.account_name")).alias("account_name"),
        F.lit("meta").alias("platform"),
        F.col("i.campaign_id").alias("campaign_id"),
        F.col("c.campaign_name").alias("campaign_name"),
        F.col("c.objective").alias("campaign_objective"),
        F.col("c.status").alias("campaign_status"),
        F.col("c.daily_budget_inr").alias("campaign_daily_budget_inr"),
        F.col("c.budget_remaining_inr").alias("campaign_budget_remaining_inr"),
        F.col("i.adset_id").alias("adset_id"),
        F.col("s.adset_name").alias("adset_name"),
        F.col("s.status").alias("adset_status"),
        F.col("s.optimization_goal"),
        F.col("s.daily_budget_inr").alias("adset_daily_budget_inr"),
        F.col("s.lifetime_budget_inr").alias("adset_lifetime_budget_inr"),
        F.col("i.ad_id").alias("ad_id"),
        F.coalesce(F.col("ad.ad_name"), F.col("i.ad_name")).alias("ad_name"),
        F.col("ad.status").alias("ad_status"),
        F.col("ad.effective_status").alias("ad_effective_status"),
        F.col("i.impressions").cast("double").alias("impressions"),
        F.col("i.reach").cast("double").alias("reach"),
        F.col("i.frequency").cast("double").alias("frequency"),
        F.col("i.clicks").cast("double").alias("clicks"),
        F.col("i.unique_clicks").cast("double").alias("unique_clicks"),
        F.col("i.inline_link_clicks").cast("double").alias("inline_link_clicks"),
        F.col("i.spend").cast("double").alias("spend_inr"),
        F.col("i.cpc").cast("double").alias("cpc"),
        F.col("i.cpm").cast("double").alias("cpm"),
        F.col("i.ctr").cast("double").alias("ctr"),
        F.current_timestamp().alias("gold_processed_at"),
    )
)
print("[META rpt]")
rpt_meta.groupBy("account_id").agg(F.count("*").alias("rows"), F.round(F.sum("spend_inr"), 2).alias("spend")).show(truncate=False)
sk = [c for c in rpt_meta.columns if c.endswith("_sk")]
if sk:
    raise RuntimeError(f"unexpected sk cols: {sk}")
write_managed_and_files(rpt_meta, "rpt_meta_ad_performance_daily", f"{GOLD_FILES_META}/rpt_meta_ad_performance_daily", ["full_date"])
print("[META DONE]")



In [ ]:
from pyspark.sql import functions as F

g_campaigns = spark.read.format("delta").load(f"{GOOGLE_SILVER}/silver_google_campaigns")
g_adgroups = spark.read.format("delta").load(f"{GOOGLE_SILVER}/silver_google_adgroups")
g_ads = spark.read.format("delta").load(f"{GOOGLE_SILVER}/silver_google_ads")
g_perf = spark.read.format("delta").load(f"{GOOGLE_SILVER}/silver_google_ad_performance")
print("[GOOGLE silver]", g_campaigns.count(), g_adgroups.count(), g_ads.count(), g_perf.count())

g_camp = g_campaigns.select("campaign_id", "campaign_name", "status", "channel_type", "daily_budget_inr").dropDuplicates(["campaign_id"])
g_ag = g_adgroups.select("adgroup_id", "adgroup_name", "status").dropDuplicates(["adgroup_id"])
g_ad = g_ads.select("ad_id", "ad_type", "status", "final_urls", "headline").dropDuplicates(["ad_id"])

rpt_google = (
    g_perf.alias("p")
    .join(g_camp.alias("c"), F.col("p.campaign_id") == F.col("c.campaign_id"), "left")
    .join(g_ag.alias("g"), F.col("p.adgroup_id") == F.col("g.adgroup_id"), "left")
    .join(g_ad.alias("ad"), F.col("p.ad_id") == F.col("ad.ad_id"), "left")
    .select(
        F.coalesce(F.col("p.platform"), F.lit("google_ads")).alias("platform"),
        F.to_date(F.col("p.date")).alias("full_date"),
        F.year(F.to_date(F.col("p.date"))).alias("year"),
        F.month(F.to_date(F.col("p.date"))).alias("month"),
        F.date_format(F.to_date(F.col("p.date")), "MMMM").alias("month_name"),
        F.date_format(F.to_date(F.col("p.date")), "EEEE").alias("day_name"),
        F.col("p.account_id").alias("account_id"),
        F.col("p.account_name").alias("account_name"),
        F.col("p.campaign_id").alias("campaign_id"),
        F.col("c.campaign_name").alias("campaign_name"),
        F.col("c.status").alias("campaign_status"),
        F.col("c.channel_type").alias("campaign_channel_or_objective"),
        F.col("c.daily_budget_inr").alias("campaign_daily_budget_inr"),
        F.col("p.adgroup_id").alias("adset_or_adgroup_id"),
        F.col("g.adgroup_name").alias("adset_or_adgroup_name"),
        F.col("g.status").alias("adset_or_adgroup_status"),
        F.col("p.ad_id").alias("ad_id"),
        F.col("ad.headline").alias("ad_name"),
        F.col("ad.ad_type").alias("ad_type"),
        F.col("ad.status").alias("ad_status"),
        F.col("ad.final_urls").alias("final_urls"),
        F.col("p.impressions").cast("double").alias("impressions"),
        F.col("p.clicks").cast("double").alias("clicks"),
        F.col("p.ctr").cast("double").alias("ctr"),
        F.col("p.spend_inr").cast("double").alias("spend_inr"),
        F.col("p.average_cpc").cast("double").alias("cpc"),
        F.col("p.conversions").cast("double").alias("conversions"),
        F.col("p.conversions_value").cast("double").alias("conversions_value"),
        F.col("p.cost_per_conversion").cast("double").alias("cost_per_conversion"),
        F.col("p.roas").cast("double").alias("roas"),
        F.lit(None).cast("double").alias("engagements"),
        F.lit(None).cast("double").alias("video_views"),
        F.current_timestamp().alias("gold_processed_at"),
    )
    .where(F.col("full_date").isNotNull())
)
print("[GOOGLE rpt]")
rpt_google.groupBy("account_id").agg(F.count("*").alias("rows"), F.round(F.sum("spend_inr"), 2).alias("spend")).show(truncate=False)
write_managed_and_files(rpt_google, "rpt_google_ad_performance_daily", f"{GOLD_FILES_GOOGLE}/rpt_google_ad_performance_daily", ["full_date"])
print("[GOOGLE DONE]")



In [ ]:
from pyspark.sql import functions as F

meta_u = spark.table(f"{S}.rpt_meta_ad_performance_daily").select(
    F.lit("meta").alias("platform"),
    "full_date", "year", "month", "month_name", "day_name",
    "account_id", "account_name",
    "campaign_id", "campaign_name", "campaign_status",
    F.col("campaign_objective").alias("campaign_channel_or_objective"),
    "campaign_daily_budget_inr",
    F.col("adset_id").alias("adset_or_adgroup_id"),
    F.col("adset_name").alias("adset_or_adgroup_name"),
    F.col("adset_status").alias("adset_or_adgroup_status"),
    "ad_id", "ad_name",
    F.lit(None).cast("string").alias("ad_type"),
    "ad_status",
    F.lit(None).cast("string").alias("final_urls"),
    F.col("impressions").cast("double"),
    F.col("clicks").cast("double"),
    F.col("ctr").cast("double"),
    F.col("spend_inr").cast("double"),
    F.col("cpc").cast("double"),
    F.lit(None).cast("double").alias("conversions"),
    F.lit(None).cast("double").alias("conversions_value"),
    F.lit(None).cast("double").alias("cost_per_conversion"),
    F.lit(None).cast("double").alias("roas"),
    F.lit(None).cast("double").alias("engagements"),
    F.lit(None).cast("double").alias("video_views"),
    "gold_processed_at",
)
google_u = spark.table(f"{S}.rpt_google_ad_performance_daily").select(
    F.col("platform").cast("string").alias("platform"),
    "full_date", "year", "month", "month_name", "day_name",
    "account_id", "account_name",
    "campaign_id", "campaign_name", "campaign_status",
    "campaign_channel_or_objective", "campaign_daily_budget_inr",
    "adset_or_adgroup_id", "adset_or_adgroup_name", "adset_or_adgroup_status",
    "ad_id", "ad_name", "ad_type", "ad_status", "final_urls",
    F.col("impressions").cast("double"),
    F.col("clicks").cast("double"),
    F.col("ctr").cast("double"),
    F.col("spend_inr").cast("double"),
    F.col("cpc").cast("double"),
    F.col("conversions").cast("double"),
    F.col("conversions_value").cast("double"),
    F.col("cost_per_conversion").cast("double"),
    F.col("roas").cast("double"),
    F.col("engagements").cast("double"),
    F.col("video_views").cast("double"),
    "gold_processed_at",
)
unified = meta_u.unionByName(google_u)
sk = [c for c in unified.columns if c.endswith("_sk")]
if sk:
    raise RuntimeError(f"Unexpected surrogate keys in unified: {sk}")
print("[UNIFIED columns]", unified.columns)
unified.groupBy("platform").agg(F.count("*").alias("rows"), F.round(F.sum("spend_inr"), 2).alias("spend")).show(truncate=False)
write_managed_and_files(unified, "rpt_unified_ad_performance", f"{GOLD_FILES_UNIFIED}/rpt_unified_ad_performance", ["platform", "full_date"])
spark.sql(f"CREATE OR REPLACE VIEW {S}.vw_unified_ad_performance AS SELECT * FROM {S}.rpt_unified_ad_performance")
spark.sql(f"CREATE OR REPLACE VIEW {S}.vw_meta_ad_performance AS SELECT * FROM {S}.rpt_meta_ad_performance_daily")
spark.sql(f"CREATE OR REPLACE VIEW {S}.vw_google_ad_performance AS SELECT * FROM {S}.rpt_google_ad_performance_daily")
spark.sql(f"SELECT platform, COUNT(*) AS rows, ROUND(SUM(spend_inr),2) AS spend FROM {S}.vw_unified_ad_performance GROUP BY platform").show(truncate=False)
spark.sql(f"SELECT platform, account_id, account_name, COUNT(*) rows FROM {S}.vw_unified_ad_performance GROUP BY platform, account_id, account_name").show(truncate=False)
print("REBUILD_BOTH_COMPLETE")

